In [ ]:
import importlib
import subprocess
import sys

REQUIRED = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'scipy': 'scipy',
    'pyvista': 'pyvista',
    'vtk': 'vtk',
    'PIL': 'pillow',
}

missing = []
for mod, pkg in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(pkg)

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
else:
    print('All required packages are already available.')


In [2]:

from pathlib import Path
import warnings

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, Normalize
from scipy.io import loadmat
from scipy.stats import gaussian_kde
import pyvista as pv

warnings.filterwarnings('ignore', category=FutureWarning)

try:
    pv.start_xvfb()
except Exception:
    pass

pv.global_theme.background = 'white'
pv.global_theme.window_size = [1600, 800]

print(f'PyVista: {pv.__version__}')


PyVista: 0.47.3


C:\Users\r4718\AppData\Local\Temp\ipykernel_14092\2958153053.py:16: PyVistaDeprecationWarning: This function is deprecated and will be removed in future version of PyVista. Use vtk with osmesa instead.
  pv.start_xvfb()



## Configuration

Adjust the paths or rendering parameters here if needed.


In [ ]:
# ================================
# 1) User configuration
# ================================

def resolve_existing_path(name: str) -> Path:
    candidates = [
        Path.cwd() / name,
        Path('/mnt/data') / name,
        Path(name),
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'Cannot find file: {name}')

INP_PATH   = resolve_existing_path('hexheartLVmeshF60S45.inp')
AHA_MAT    = resolve_existing_path('AHALVMeshDivisionPCAReconstructed.mat')
STRAIN_MAT = resolve_existing_path('strainComparison_Baseline.mat')

OUTPUT_DIR = Path('merged_lv_visualization_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_VTU = True

# =====================================================================
# [NEW] Reference-configuration contract
# =====================================================================
# The MAT file stores the forward map
#     early diastole (unloaded, X)  -->  end diastole (loaded, x),
# i.e. `node_ori` = X, `dis` = x - X, and `strain_tensor_cra` = E, the
# Green-Lagrange strain of that forward map, referred to X.
#
# This notebook now reports strain with the END-DIASTOLIC configuration as
# the reference, i.e. the inverse map x --> X obtained from F^{-1}.
#
#   REFERENCE_CONFIG = 'ED'   : all geometry is drawn on the ED mesh
#   STRAIN_MEASURE_OUT:
#     'inverse_gl' : E* = 1/2 (F^{-T} F^{-1} - I) = 1/2 (C^{-1} - I)
#                    Green-Lagrange strain of the ED -> early-diastole map,
#                    referred to ED.  Circumferential/longitudinal components
#                    are NEGATIVE (the wall shortens on unloading), matching
#                    the clinical convention in which ED is the zero-strain
#                    state.  <-- default, this is what was requested.
#     'almansi'    : e = 1/2 (I - B^{-1}) = -E*, i.e. the SAME forward
#                    deformation (early diastole -> ED) but measured per unit
#                    ED length.  Signs match the original figures.
REFERENCE_CONFIG    = 'ED'
STRAIN_MEASURE_OUT  = 'inverse_gl'     # 'inverse_gl' | 'almansi'

# How the ED-frame components are obtained:
#   'corotational' : algebraic, uses only `strain_tensor_cra`.
#                    Exact identity  R^T (F^{-T}F^{-1}) R = C^{-1}, so the
#                    ED-referred components in the *co-rotated* local (c,r,l)
#                    triad follow from C = 2E + I alone.  No extra assumption
#                    about how the anatomical axes were built in MATLAB.
#   'geometric_ed' : independent route. F is recomputed element-wise from the
#                    mesh, and the components are taken in a (c,r,l) triad
#                    rebuilt geometrically ON THE ED MESH.  Use this to
#                    cross-validate; it does not rely on the Voigt ordering.
STRAIN_AXES_MODE = 'corotational'

# Voigt layout of `strain_tensor_cra` (6 columns).  Columns 0-2 are the
# normal components (verified in the original notebook).  The shear layout is
# NOT verifiable from the file alone; the two common conventions are
#   'abaqus'  -> (11, 22, 33, 12, 13, 23)
#   'voigt'   -> (11, 22, 33, 23, 13, 12)
# The audit cell below reports det(F) vs sqrt(det C) for both; pick the one
# that agrees. Set USE_SHEAR_TERMS = False to fall back to a diagonal-only
# inversion (valid when the off-diagonal strains are small).
STRAIN_SHEAR_LAYOUT = 'abaqus'
USE_SHEAR_TERMS     = True

# --- Rendering -----------------------------------------------------------
ZOOM          = 1.08
PARALLEL_PROJ = True
CLIP_NORMAL   = (-1.0, 0.0, 0.0)   # cutaway shows the back half of the LV

# =====================================================================
# [NEW] Journal figure specification (OUP / JRSS-C)
# =====================================================================
#  * one file per figure, all panels of a multipanel figure on one page
#  * panel letters A, B, C ... in the upper-left corner of each panel
#  * NO figure title, NO legend/key inside the image -> emitted separately to
#    `figure_captions_and_alt_text.md` for pasting into the manuscript
#  * raster (3-D renderings): TIFF, >= 350 dpi for shaded images,
#    600 dpi used here so the same files also satisfy the line-art rule
#  * line art (KDE / boxplots): vector PDF, plus a 600 dpi TIFF fallback
FIG_WIDTH_MM      = 170.0     # full text width; use 85.0 for a single column
DPI_RASTER        = 600
DPI_LINEART       = 600
MAX_RENDER_PX     = 4200      # cap on the off-screen GL window width
TIFF_COMPRESSION  = 'tiff_lzw'
SAVE_VECTOR_COPY  = True      # also write .pdf for pure line art
PANEL_LETTERS     = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
PANEL_LETTER_SIZE = 11
DIRECT_REGION_LABELS = True   # print region names on the panel instead of a key

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'axes.linewidth': 0.6,
    'pdf.fonttype': 42,       # embed TrueType, keeps text selectable/editable
    'ps.fonttype': 42,
    'savefig.facecolor': 'white',
})

# --- Surface appearance --------------------------------------------------
DEFAULT_SURFACE_COLOR = '#E8E8E8'
WIRE_COLOR = '#FFFFFF'
WIRE_ALPHA = 0.18
WIRE_WIDTH = 0.35

OUTLINE_COLOR = '#000000'
OUTLINE_WIDTH = 2.6

AHA17_EDGE_COLOR = '#1A1A1A'
AHA17_EDGE_WIDTH = 2.2
AHA17_EDGE_ALPHA = 0.98

AHA5_EDGE_COLOR = '#000000'
AHA5_EDGE_WIDTH = 3.2
AHA5_EDGE_ALPHA = 1.00

DIV_CMAP = plt.get_cmap('RdBu_r')

# [P3] Strain colour-limit controls (unchanged logic).
STRAIN_CLIM_PCT       = (5.0, 95.0)
STRAIN_BASAL_EXCLUDE  = 0.05

# [P2] Column contract of `strain_tensor_cra` for the NORMAL components.
#     col0 mean +0.119  (circumferential stretch at ED: expected +)
#     col1 mean -0.131  (radial thinning at ED: expected -)
#     col2 mean +0.156  (longitudinal stretch at ED: expected +)
STRAIN_COLS = {
    'circ': 0,   # E_cc
    'rad':  1,   # E_rr
    'lon':  2,   # E_ll
}

# Long axis of the mesh, used only by STRAIN_AXES_MODE = 'geometric_ed'.
LONG_AXIS = (0.0, 0.0, 1.0)



## Label definitions and remapped AHA5 grouping


In [4]:

# ================================
# 2) AHA label definitions
# ================================

# Keep the existing AHA17 field mapping unchanged.
AHA17_ELEM_FIELDS = [
    ('elem_basa_InfSept', 1), ('elem_basa_AntSept', 2), ('elem_basa_Ant',    3),
    ('elem_base_AntLat',  4), ('elem_base_InfLat',  5), ('elem_base_Inf',    6),
    ('elem_midd_InfSept', 7), ('elem_midd_AntSept', 8), ('elem_midd_Ant',    9),
    ('elem_midd_AntLat', 10), ('elem_midd_InfLat', 11), ('elem_midd_Inf',   12),
    ('elem_apex_Sept',   13), ('elem_apex_Ant',    14), ('elem_apex_Lat',   15),
    ('elem_apex_Inf',    16), ('elem_apicalRegion',17),
]

AHA17_LABELS = {
    1: 'Basal inferoseptal',
    2: 'Basal anteroseptal',
    3: 'Basal anterior',
    4: 'Basal anterolateral',
    5: 'Basal inferolateral',
    6: 'Basal inferior',
    7: 'Mid inferoseptal',
    8: 'Mid anteroseptal',
    9: 'Mid anterior',
    10: 'Mid anterolateral',
    11: 'Mid inferolateral',
    12: 'Mid inferior',
    13: 'Apical septal',
    14: 'Apical anterior',
    15: 'Apical lateral',
    16: 'Apical inferior',
    17: 'Apical cap / conflict',
}

# Remapped AHA5 grouping based on the preserved AHA17 numbering above.
AHA17_TO_AHA5 = {
    3: 1, 9: 1,                       # Anterior
    4: 2, 5: 2, 10: 2, 11: 2,         # Lateral
    6: 3, 12: 3,                      # Inferior
    1: 4, 2: 4, 7: 4, 8: 4,           # Septal
    13: 5, 14: 5, 15: 5, 16: 5, 17: 5 # Apical
}

AHA5_GROUP_NAMES = {
    1: 'Anterior',
    2: 'Lateral',
    3: 'Inferior',
    4: 'Septal',
    5: 'Apical',
}

AHA5_COLORS = [
    '#0072B2',  # Anterior
    '#009E73',  # Lateral
    '#E69F00',  # Inferior
    '#CC79A7',  # Septal
    '#D55E00',  # Apical
]
AHA5_CMAP = ListedColormap(AHA5_COLORS, name='aha5')

LATERAL_SEGMENTS = (4, 5, 10, 11)
LATERAL_SUB_COLORS = ['#004D40', '#1B9E77', '#4DB6AC', '#80CBC4']
LATERAL_SUB_CMAP = ListedColormap(LATERAL_SUB_COLORS, name='lateral4')

STRAIN_LABELS = {
    'circ': 'Circumferential strain',
    'rad': 'Radial strain',
    'lon': 'Longitudinal strain',
}



## I/O and mesh construction helpers


In [5]:

# ================================
# 3) I/O helpers
# ================================

def _as_1d_int(x):
    a = np.asarray(x).reshape(-1)
    if a.size and np.issubdtype(a.dtype, np.number):
        a = a[np.isfinite(a)]
    return a.astype(np.int64)


def parse_abaqus_inp(inp_file):
    """Parse *Node and *Element(type=C3D8/C3D8H) from an Abaqus .inp file."""
    node_ids, points, conn = [], [], []
    in_nodes = False
    in_hex = False

    with open(inp_file, 'r', encoding='utf-8', errors='ignore') as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('**'):
                continue

            if line.startswith('*'):
                up = line.upper()
                in_nodes = up.startswith('*NODE')
                is_elem = up.startswith('*ELEMENT')
                in_hex = is_elem and ('TYPE=C3D8' in up or 'TYPE=C3D8H' in up)
                continue

            parts = [p.strip() for p in line.split(',') if p.strip()]
            if in_nodes and len(parts) >= 4:
                node_ids.append(int(parts[0]))
                points.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif in_hex and len(parts) >= 9:
                conn.append([int(x) for x in parts[1:9]])

    if not node_ids or not conn:
        raise RuntimeError('Failed to parse nodes or hex elements from the INP file.')

    node_ids = np.asarray(node_ids, dtype=np.int64)
    points = np.asarray(points, dtype=np.float64)
    conn_1b = np.asarray(conn, dtype=np.int64)

    lookup = np.full(node_ids.max() + 2, -1, dtype=np.int64)
    lookup[node_ids] = np.arange(node_ids.size)
    conn_0b = lookup[conn_1b]
    if (conn_0b < 0).any():
        raise RuntimeError('Some elements reference undefined node IDs.')

    return points, conn_0b


def load_aha_labels(mat_file, n_cells):
    """Load AHA17 labels from the MATLAB struct and remap them to AHA5."""
    mat = loadmat(mat_file, squeeze_me=True, struct_as_record=False)
    if 'AHALVMeshDivision' not in mat:
        raise KeyError("Missing 'AHALVMeshDivision' in the AHA MATLAB file.")

    s = mat['AHALVMeshDivision']

    def _get(name):
        if hasattr(s, name):
            return getattr(s, name)
        return None

    node_regions = _as_1d_int(_get('nodeRegions'))
    if node_regions.size == 0:
        raise ValueError('Missing nodeRegions in the AHA MATLAB file.')

    aha17 = np.zeros(n_cells, dtype=np.int32)
    hit_count = np.zeros(n_cells, dtype=np.int32)
    missing_fields = []

    for field, rid in AHA17_ELEM_FIELDS:
        ids = _get(field)
        if ids is None:
            missing_fields.append(field)
            continue
        idx = np.unique(_as_1d_int(ids) - 1)
        if (idx < 0).any() or (idx >= n_cells).any():
            raise ValueError(f'{field}: element IDs are out of range.')
        first_hit = hit_count[idx] == 0
        aha17[idx[first_hit]] = rid
        hit_count[idx] += 1

    if missing_fields:
        raise ValueError(f'Missing MATLAB fields: {missing_fields}')
    if (hit_count == 0).any():
        raise ValueError(f'{int((hit_count == 0).sum())} elements are not covered by any elem_* field.')

    conflict_mask = hit_count > 1
    aha17[conflict_mask] = 17

    aha5 = np.array([AHA17_TO_AHA5[int(x)] for x in aha17], dtype=np.int32)
    return node_regions, aha17, aha5, conflict_mask


def load_strain_struct(mat_file, n_cells_expected=None, n_points_expected=None):
    """Load strain tensor and nodal displacement from strainComparison_Baseline.mat."""
    mat = loadmat(mat_file, squeeze_me=True, struct_as_record=False)
    if 'strainComparison' not in mat:
        raise KeyError("Missing 'strainComparison' in the strain MATLAB file.")

    s = mat['strainComparison']
    T = np.asarray(getattr(s, 'strain_tensor_cra'), dtype=np.float64)
    dis = np.asarray(getattr(s, 'dis'), dtype=np.float64)
    node_ori = np.asarray(getattr(s, 'node_ori'), dtype=np.float64)

    if n_cells_expected is not None and T.shape[0] != n_cells_expected:
        raise ValueError('strain_tensor_cra does not match the mesh cell count.')
    if n_points_expected is not None and dis.shape[0] != n_points_expected:
        raise ValueError('dis does not match the mesh point count.')
    if n_points_expected is not None and node_ori.shape[0] != n_points_expected:
        raise ValueError('node_ori does not match the mesh point count.')

    return {
        'strain_tensor': T,
        'dis': dis,
        'node_ori': node_ori,
    }


def build_grid(points, conn, node_regions, aha17, aha5, strain_tensor=None, dis=None):
    n_cells = conn.shape[0]
    cells = np.hstack([np.full((n_cells, 1), 8, dtype=np.int64), conn]).ravel()
    celltypes = np.full(n_cells, pv.CellType.HEXAHEDRON, dtype=np.uint8)
    grid = pv.UnstructuredGrid(cells, celltypes, points)

    grid.cell_data['AHA17'] = aha17.astype(np.int32)
    grid.cell_data['AHA5'] = aha5.astype(np.int32)

    # Precomputed RGB so rendering bypasses VTK's IndexedLookup, which otherwise
    # remaps colours when a sub-mesh (e.g. a short-axis slab) is missing some
    # AHA5 categories. See the "second patch" note at the top of this notebook.
    rgb = np.zeros((aha5.size, 3), dtype=np.float32)
    palette_rgb = np.array([
        [float(int(c[1:3], 16)), float(int(c[3:5], 16)), float(int(c[5:7], 16))]
        for c in AHA5_COLORS
    ], dtype=np.float32) / 255.0
    for gid in range(1, 6):
        m = aha5 == gid
        rgb[m] = palette_rgb[gid - 1]
    grid.cell_data['AHA5_rgb'] = rgb

    if node_regions.size == points.shape[0]:
        grid.point_data['node_regions'] = node_regions.astype(np.int32)

    if strain_tensor is not None:
        for key, idx in STRAIN_COLS.items():
            grid.cell_data[f'strain_{key}'] = strain_tensor[:, idx].astype(np.float64)

    if dis is not None:
        grid.point_data['disp_x'] = dis[:, 0]
        grid.point_data['disp_y'] = dis[:, 1]
        grid.point_data['disp_z'] = dis[:, 2]
        grid.point_data['disp_mag'] = np.linalg.norm(dis, axis=1)

    return grid


def fix_septal_inferior_boundary(aha17, aha5, cell_centers,
                                  inf_theta_min=30.0, inf_theta_max=90.0,
                                  verbose=True):
    """Data-cleaning step for the Septal-Inferior boundary region.

    The source MAT file contains a non-trivial number of cells whose AHA17
    label places them in the Anterior (3, 9) or Lateral (4, 5, 10, 11)
    segments, but whose actual spatial position in the short-axis (x, y)
    plane falls inside the Inferior angular band occupied by seg 6 / 12.
    These cells are mislabelled (e.g. seg 4/10 cells at theta ~ +81 deg,
    which is anatomically opposite to AntLat's correct ~ -45 deg).

    This function identifies such cells by angular position and reassigns
    them to the Inferior segment — seg 6 if the cell sits in the basal
    half-slab (current AHA17 in 1..6) and seg 12 if in the mid half-slab
    (current AHA17 in 7..12). AHA5 is updated to Inferior (3) accordingly.
    Apical cells (AHA17 in 13..17) are left untouched.

    Returns (aha17_fixed, aha5_fixed, n_reassigned).
    """
    cc = np.asarray(cell_centers)
    cx, cy = cc[:, 0].mean(), cc[:, 1].mean()
    theta = np.degrees(np.arctan2(cc[:, 1] - cy, cc[:, 0] - cx))

    is_basal_mid = (aha17 >= 1) & (aha17 <= 12)
    is_basal     = (aha17 >= 1) & (aha17 <= 6)
    is_mid       = (aha17 >= 7) & (aha17 <= 12)
    in_zone      = (theta >= inf_theta_min) & (theta <= inf_theta_max)
    is_ant_lat   = (aha5 == 1) | (aha5 == 2)

    to_fix = is_basal_mid & in_zone & is_ant_lat

    new_aha17 = aha17.copy()
    new_aha5  = aha5.copy()
    new_aha17[to_fix & is_basal] = 6
    new_aha17[to_fix & is_mid]   = 12
    new_aha5[to_fix] = 3

    if verbose:
        n = int(to_fix.sum())
        print(f'Septal-Inferior boundary fix: reassigned {n} cells '
              f'from Anterior/Lateral to Inferior '
              f'(angular zone theta in [{inf_theta_min:+.0f}, {inf_theta_max:+.0f}] deg).')
        for orig_seg in (3, 4, 5, 9, 10, 11):
            m = to_fix & (aha17 == orig_seg)
            if m.any():
                print(f'  from AHA17 seg {orig_seg:2d}: {int(m.sum()):4d} cells')

    return new_aha17, new_aha5, int(to_fix.sum())



## Kinematics: inverse deformation gradient and the ED-referred strain

$\mathbf{F}$ is recomputed element-wise from the mesh (trilinear hexahedral shape functions evaluated at the element centroid), and the ED-referred strain is obtained either algebraically from the stored tensor via $\mathbf{C}^{-1}$ or independently from $\mathbf{B}^{-1}$ on the ED geometry.


In [ ]:
# ==========================================================================
# 3b) Kinematics: end-diastolic reference configuration
# ==========================================================================
# Notation
#   X   node coordinates in early diastole (unloaded reference of the solver)
#   x   node coordinates at end diastole,  x = X + dis
#   F   = dx/dX,   C = F^T F = 2E + I,   B = F F^T
#   E   Green-Lagrange strain of X -> x, referred to X   (what the MAT stores)
#
# Taking ED as the reference configuration and mapping ED -> early diastole
# means using the inverse motion, whose deformation gradient is f = F^{-1}.
# Its Green-Lagrange strain is
#
#   E* = 1/2 (f^T f - I) = 1/2 (F^{-T} F^{-1} - I) = 1/2 (B^{-1} - I) = -e ,
#
# with e = 1/2 (I - B^{-1}) the Almansi-Euler strain.
#
# KEY IDENTITY (why no extra data are needed).  With the polar decomposition
# F = R U, the components of B^{-1} in the basis R g_i (the local anatomical
# triad carried along by the rigid rotation of the material) are
#
#   R^T B^{-1} R = R^T (R U^{-2} R^T) R = U^{-2} = C^{-1} .
#
# Hence, in the co-rotated (c, r, l) triad,
#
#   E* = 1/2 (C^{-1} - I),      C = 2E + I,
#
# which is a purely algebraic, element-wise 3x3 inversion of the strain tensor
# already stored in the MAT file. `geometric_ed` below is an independent
# recomputation from the mesh, used as a cross-check.

HEX_XI = np.array([
    [-1., -1., -1.], [1., -1., -1.], [1., 1., -1.], [-1., 1., -1.],
    [-1., -1.,  1.], [1., -1.,  1.], [1., 1.,  1.], [-1., 1.,  1.],
])                                   # Abaqus C3D8 node ordering
HEX_DNDXI_CENTROID = HEX_XI / 8.0    # dN_a/dxi_j evaluated at xi = 0


def deformation_gradient_hex(points_ref, conn, dis):
    """Element-centroid deformation gradient F for trilinear hexahedra.

    Returns (n_cells, 3, 3). Exact for homogeneous deformation; at the
    centroid of a general C3D8 element it is the standard one-point
    (reduced-integration) evaluation.
    """
    dN = HEX_DNDXI_CENTROID
    Xe = points_ref[conn]                       # (nc, 8, 3)
    xe = (points_ref + dis)[conn]               # (nc, 8, 3)
    J0 = np.einsum('eai,aj->eij', Xe, dN)       # dX_i/dxi_j
    dNdX = np.einsum('aj,eji->eai', dN, np.linalg.inv(J0))
    F = np.einsum('eai,eaj->eij', xe, dNdX)
    return F


def voigt_to_tensor(V, normal_cols=(0, 1, 2), layout='abaqus', use_shear=True):
    """(n, 6) Voigt strain -> (n, 3, 3) symmetric tensor.

    layout='abaqus' : columns 3,4,5 are the 12, 13, 23 components
    layout='voigt'  : columns 3,4,5 are the 23, 13, 12 components
    Off-diagonal columns are treated as TENSOR components (E_ij, not 2E_ij).
    """
    V = np.asarray(V, dtype=np.float64)
    n = V.shape[0]
    T = np.zeros((n, 3, 3), dtype=np.float64)
    T[:, 0, 0] = V[:, normal_cols[0]]
    T[:, 1, 1] = V[:, normal_cols[1]]
    T[:, 2, 2] = V[:, normal_cols[2]]
    if use_shear and V.shape[1] >= 6:
        pairs = ((0, 1), (0, 2), (1, 2)) if layout == 'abaqus' else ((1, 2), (0, 2), (0, 1))
        for k, (i, j) in enumerate(pairs):
            T[:, i, j] = V[:, 3 + k]
            T[:, j, i] = V[:, 3 + k]
    return T


def right_cauchy_green(E):
    """C = 2E + I from the Green-Lagrange strain."""
    return 2.0 * np.asarray(E, dtype=np.float64) + np.eye(3)


def ed_referenced_strain(E):
    """E* = 1/2 (C^{-1} - I): Green-Lagrange strain of the ED -> early-diastole
    map, expressed in the co-rotated local triad. Returns (n, 3, 3)."""
    C = right_cauchy_green(E)
    Cinv = np.linalg.inv(C)
    return 0.5 * (Cinv - np.eye(3))


def almansi_strain(E):
    """e = 1/2 (I - B^{-1}) = -E*, same forward deformation, ED length scale."""
    return -ed_referenced_strain(E)


def local_triad(cell_centers, long_axis=LONG_AXIS):
    """Geometric (circumferential, radial, longitudinal) orthonormal triad.

    The longitudinal direction is the LV long axis; the radial direction is
    the outward normal to that axis through the centroid of the mesh; the
    circumferential direction completes a right-handed frame.
    Returns Q with Q[e, :, 0] = c, Q[e, :, 1] = r, Q[e, :, 2] = l.
    """
    cc = np.asarray(cell_centers, dtype=np.float64)
    a = np.asarray(long_axis, dtype=np.float64)
    a = a / np.linalg.norm(a)
    axis_point = cc.mean(axis=0)
    d = cc - axis_point
    r = d - np.outer(d @ a, a)
    nrm = np.linalg.norm(r, axis=1, keepdims=True)
    nrm[nrm < 1e-12] = 1.0
    r = r / nrm
    c = np.cross(np.broadcast_to(a, r.shape), r)
    c = c / np.linalg.norm(c, axis=1, keepdims=True)
    Q = np.stack([c, r, np.broadcast_to(a, r.shape)], axis=2)
    return Q


def ed_strain_from_mesh(F, cell_centers_ed, long_axis=LONG_AXIS,
                        measure='inverse_gl'):
    """Independent route: E* (or e) in a triad rebuilt on the ED geometry."""
    Binv = np.linalg.inv(np.einsum('eij,ekj->eik', F, F))       # (F F^T)^{-1}
    Estar = 0.5 * (Binv - np.eye(3))
    T = Estar if measure == 'inverse_gl' else -Estar
    Q = local_triad(cell_centers_ed, long_axis)
    return np.einsum('eia,eij,ejb->eab', Q, T, Q)


def ed_strain_components(strain_tensor, F, cell_centers_ed,
                         mode=STRAIN_AXES_MODE,
                         measure=STRAIN_MEASURE_OUT,
                         layout=STRAIN_SHEAR_LAYOUT,
                         use_shear=USE_SHEAR_TERMS):
    """Return {'circ', 'rad', 'lon'} normal components in the ED frame."""
    if mode == 'corotational':
        E = voigt_to_tensor(
            strain_tensor,
            normal_cols=(STRAIN_COLS['circ'], STRAIN_COLS['rad'], STRAIN_COLS['lon']),
            layout=layout, use_shear=use_shear,
        )
        T = ed_referenced_strain(E) if measure == 'inverse_gl' else almansi_strain(E)
    elif mode == 'geometric_ed':
        T = ed_strain_from_mesh(F, cell_centers_ed, measure=measure)
    else:
        raise ValueError(f'Unknown STRAIN_AXES_MODE: {mode}')
    return {'circ': T[:, 0, 0], 'rad': T[:, 1, 1], 'lon': T[:, 2, 2]}


def audit_kinematics(strain_tensor, F, cell_centers_ed):
    """Consistency report. Run this before trusting any figure.

    1. det F  vs  sqrt(det C) from the stored strain tensor. These must agree
       element-wise if the Voigt layout and the Green-Lagrange assumption are
       both correct; the check is repeated for the two shear layouts.
    2. Normal components of E* from the two independent routes.
    """
    detF = np.linalg.det(F)
    print(f'Jacobian from the mesh:  det F  mean={detF.mean():.4f}  '
          f'range=[{detF.min():.4f}, {detF.max():.4f}]   (min must be > 0)')

    best = None
    for layout in ('abaqus', 'voigt'):
        for use_shear in (True, False):
            E = voigt_to_tensor(
                strain_tensor,
                normal_cols=(STRAIN_COLS['circ'], STRAIN_COLS['rad'], STRAIN_COLS['lon']),
                layout=layout, use_shear=use_shear,
            )
            detC = np.linalg.det(right_cauchy_green(E))
            ok = detC > 0
            J_from_E = np.full_like(detC, np.nan)
            J_from_E[ok] = np.sqrt(detC[ok])
            err = np.nanmedian(np.abs(J_from_E - detF) / np.abs(detF))
            tag = f'layout={layout:6s} shear={"on " if use_shear else "off"}'
            print(f'  {tag}:  median |J_E - det F| / det F = {err:.4%}')
            if best is None or err < best[0]:
                best = (err, layout, use_shear)
    print(f'  -> best agreement: layout={best[1]}, shear={"on" if best[2] else "off"} '
          f'({best[0]:.4%}). Set STRAIN_SHEAR_LAYOUT / USE_SHEAR_TERMS accordingly.')

    co = ed_strain_components(strain_tensor, F, cell_centers_ed, mode='corotational')
    ge = ed_strain_components(strain_tensor, F, cell_centers_ed, mode='geometric_ed')
    print('\nED-referenced normal strains, two independent routes '
          f'(measure = {STRAIN_MEASURE_OUT}):')
    print(f'{"":>6s}  {"co-rotational":>26s}  {"geometric (ED axes)":>26s}  {"median |diff|":>14s}')
    for key in ('circ', 'rad', 'lon'):
        a, b = co[key], ge[key]
        print(f'{key:>6s}  mean {a.mean():+8.4f} sd {a.std():6.4f}  '
              f'mean {b.mean():+8.4f} sd {b.std():6.4f}  {np.median(np.abs(a - b)):14.4f}')
    print('A residual difference is expected: the co-rotational frame is the '
          'early-diastolic anatomical triad carried by R, whereas the geometric '
          'frame is rebuilt on the ED geometry. Large disagreement (same order '
          'as the strains themselves) indicates a wrong Voigt layout or a '
          'strain measure other than Green-Lagrange.')
    return co, ge



## Visualisation helpers


In [ ]:
# ================================
# 4) Visualisation helpers
# ================================
# All journal-facing rules live here:
#   - no titles, no legends/keys and no panel captions inside the image file
#   - panel letters A, B, C ... drawn in the upper-left corner of each panel
#   - exact physical width (FIG_WIDTH_MM) and resolution (DPI_RASTER)
#   - captions and alt text are accumulated in FIGURE_REGISTRY and written to
#     a separate markdown file at the end of the notebook

FIGURE_REGISTRY = []


def mm_to_in(mm):
    return float(mm) / 25.4


def render_window_size(n_panels, panel_aspect=1.0):
    """Off-screen GL window large enough that the composite still meets DPI.

    panel_aspect = panel height / panel width.
    """
    target_w = int(round(mm_to_in(FIG_WIDTH_MM) * DPI_RASTER))
    w = min(target_w, MAX_RENDER_PX)
    h = int(round(w * panel_aspect / n_panels))
    return (w, h)


def extract_surface(dataset):
    surf = dataset.extract_surface(algorithm='dataset_surface').clean()
    surf = surf.triangulate().clean()
    return surf


def extract_label_boundaries(surface, label):
    if label not in surface.cell_data:
        return None

    pieces = []
    for rid in np.unique(surface.cell_data[label]):
        ids = np.flatnonzero(surface.cell_data[label] == rid)
        if ids.size == 0:
            continue
        part = surface.extract_cells(ids)
        edges = part.extract_feature_edges(
            boundary_edges=True,
            non_manifold_edges=False,
            feature_edges=False,
            manifold_edges=False,
        )
        if edges.n_cells > 0:
            pieces.append(edges)

    if not pieces:
        return None

    out = pieces[0]
    for p in pieces[1:]:
        out = out.merge(p)
    return out.clean()


def compute_camera(bounds, view='oblique'):
    """view='oblique' : long-axis 3/4 view; view='short_axis' : from the base."""
    xmin, xmax, ymin, ymax, zmin, zmax = bounds
    center = np.array([(xmin + xmax) / 2, (ymin + ymax) / 2, (zmin + zmax) / 2])
    diag = np.linalg.norm([xmax - xmin, ymax - ymin, zmax - zmin])
    if view == 'short_axis':
        pos = center + np.array([0.0, 0.0, 1.9]) * diag
        up = (0.0, 1.0, 0.0)
    else:
        pos = center + np.array([1.35, -1.85, 1.05]) * diag
        up = (0.0, 0.0, 1.0)
    return [tuple(pos), tuple(center), up]


def add_edges(pl, edges, color, width, alpha=1.0):
    if edges is None or edges.n_cells == 0:
        return
    pl.add_mesh(edges, color=color, line_width=width, opacity=alpha, lighting=False)


def add_plain_surface(pl, mesh, color=DEFAULT_SURFACE_COLOR, opacity=1.0, show_wire=False):
    pl.add_mesh(
        mesh, color=color, smooth_shading=True, opacity=opacity,
        ambient=0.32, diffuse=0.70, specular=0.05,
        silhouette=dict(color=OUTLINE_COLOR, line_width=OUTLINE_WIDTH),
    )
    if show_wire:
        pl.add_mesh(mesh, style='wireframe', color=WIRE_COLOR, opacity=WIRE_ALPHA,
                    line_width=WIRE_WIDTH, lighting=False)


def add_categorical_surface(pl, mesh, scalars, cmap, clim, opacity=1.0):
    """Discrete cell labels via a precomputed RGB array (bypasses IndexedLookup)."""
    rgb_name = f'{scalars}_rgb'
    if rgb_name in mesh.cell_data:
        pl.add_mesh(
            mesh, scalars=rgb_name, preference='cell', rgb=True,
            show_scalar_bar=False, smooth_shading=True, opacity=opacity,
            ambient=0.32, diffuse=0.70, specular=0.05,
            silhouette=dict(color=OUTLINE_COLOR, line_width=OUTLINE_WIDTH),
        )
    else:
        pl.add_mesh(
            mesh, scalars=scalars, preference='cell', cmap=cmap, clim=clim,
            categories=True, show_scalar_bar=False, interpolate_before_map=False,
            smooth_shading=True, opacity=opacity,
            ambient=0.32, diffuse=0.70, specular=0.05,
            silhouette=dict(color=OUTLINE_COLOR, line_width=OUTLINE_WIDTH),
        )


def add_continuous_surface(pl, mesh, scalars, clim, opacity=1.0):
    pl.add_mesh(
        mesh, scalars=scalars, preference='cell', cmap=DIV_CMAP, clim=clim,
        show_scalar_bar=False, interpolate_before_map=True, smooth_shading=True,
        opacity=opacity, ambient=0.32, diffuse=0.70, specular=0.05,
        silhouette=dict(color=OUTLINE_COLOR, line_width=OUTLINE_WIDTH),
    )


def add_region_labels(pl, dataset, label='AHA5', names=None, font_size=22):
    """Direct on-panel region names, so no colour key is needed in the figure."""
    if not DIRECT_REGION_LABELS or label not in dataset.cell_data:
        return
    names = names or AHA5_GROUP_NAMES
    cc = dataset.cell_centers().points
    ids = np.asarray(dataset.cell_data[label])
    pts, txt = [], []
    for rid in np.unique(ids):
        m = ids == rid
        if not m.any():
            continue
        pts.append(cc[m].mean(axis=0) * 1.06)
        txt.append(str(names.get(int(rid), int(rid))))
    if not pts:
        return
    pl.add_point_labels(
        np.asarray(pts), txt, font_size=font_size, text_color='black',
        shape=None, show_points=False, always_visible=True, bold=False,
    )


def _render_panel(pl, dataset, draw_fn, camera):
    """No in-render text: panel letters are added by the matplotlib composer."""
    if PARALLEL_PROJ:
        pl.enable_parallel_projection()
    draw_fn(pl, dataset)
    pl.camera_position = camera
    pl.reset_camera()
    pl.camera.zoom(ZOOM)


def dual_view_image(dataset, draw_fn, window_size=None, panel_aspect=1.0):
    """Panel A: exterior. Panel B: long-axis cutaway."""
    center = np.array(dataset.center)
    clipped = dataset.clip(normal=CLIP_NORMAL, origin=center, invert=False)
    camera = compute_camera(dataset.bounds, view='oblique')

    window_size = window_size or render_window_size(2, panel_aspect)
    pl = pv.Plotter(shape=(1, 2), off_screen=True, window_size=window_size, border=False)
    pl.set_background('white')
    for j, ds in enumerate([dataset, clipped]):
        pl.subplot(0, j)
        _render_panel(pl, ds, draw_fn, camera)
    img = pl.screenshot(return_img=True)
    pl.close()
    return np.asarray(img)


def triple_view_image(dataset, draw_fn, window_size=None, panel_aspect=1.05,
                      label_fn=None):
    """Panel A: exterior. Panel B: long-axis cutaway. Panel C: short-axis slab."""
    center = np.array(dataset.center)
    bounds = dataset.bounds
    zmax, zmin = bounds[5], bounds[4]
    slab_hi = zmax - 0.08 * (zmax - zmin)
    slab_lo = zmax - 0.35 * (zmax - zmin)
    short_axis = dataset.clip(normal=(0, 0, 1), origin=(center[0], center[1], slab_lo), invert=False)
    short_axis = short_axis.clip(normal=(0, 0, -1), origin=(center[0], center[1], slab_hi), invert=False)
    long_cut = dataset.clip(normal=CLIP_NORMAL, origin=center, invert=False)

    cam_oblique = compute_camera(bounds, view='oblique')
    cam_short = compute_camera(bounds, view='short_axis')

    window_size = window_size or render_window_size(3, panel_aspect)
    pl = pv.Plotter(shape=(1, 3), off_screen=True, window_size=window_size, border=False)
    pl.set_background('white')
    specs = [(dataset, cam_oblique), (long_cut, cam_oblique), (short_axis, cam_short)]
    for j, (ds, cam) in enumerate(specs):
        pl.subplot(0, j)
        _render_panel(pl, ds, draw_fn, cam)
        if label_fn is not None:
            label_fn(pl, ds, j)
    img = pl.screenshot(return_img=True)
    pl.close()
    return np.asarray(img)


def _save_current(fig, out_stem, vector=False):
    """Write TIFF (LZW, tagged DPI) and optionally a vector PDF."""
    paths = []
    tif = Path(out_stem).with_suffix('.tiff')
    dpi = DPI_LINEART if vector else DPI_RASTER
    fig.savefig(tif, dpi=dpi, facecolor='white',
                pil_kwargs={'compression': TIFF_COMPRESSION})
    try:                                    # enforce the resolution tag
        from PIL import Image
        with Image.open(tif) as im:
            im.save(tif, format='TIFF', dpi=(dpi, dpi), compression=TIFF_COMPRESSION)
    except Exception as exc:
        print(f'  (warning: could not rewrite DPI tag: {exc})')
    paths.append(tif)
    if vector and SAVE_VECTOR_COPY:
        pdf = Path(out_stem).with_suffix('.pdf')
        fig.savefig(pdf, facecolor='white')
        paths.append(pdf)
    for p in paths:
        print(f'Saved: {p}')
    return paths


def register_figure(number, stem, paths, title, legend, alt_text):
    FIGURE_REGISTRY.append({
        'number': number, 'stem': str(stem),
        'files': [str(p.name) for p in paths],
        'title': title, 'legend': legend, 'alt': alt_text,
    })


def compose_panels(img, out_stem, n_panels, colorbar=None,
                   number=None, title='', legend='', alt_text=''):
    """Lay a rendered strip out as one page, add panel letters, save, register.

    The image file contains ONLY the panels, the letters and (where relevant)
    a labelled colour scale. Title and legend text are written to the caption
    file instead, as required by the journal.
    """
    h, w = img.shape[:2]
    fig_w_in = mm_to_in(FIG_WIDTH_MM)
    img_h_in = fig_w_in * h / w
    cbar_h_in = 0.42 if colorbar is not None else 0.0

    fig = plt.figure(figsize=(fig_w_in, img_h_in + cbar_h_in), facecolor='white')
    ax_img = fig.add_axes([0.0, cbar_h_in / (img_h_in + cbar_h_in), 1.0,
                           img_h_in / (img_h_in + cbar_h_in)])
    interp = 'none' if w >= fig_w_in * DPI_RASTER else 'lanczos'
    ax_img.imshow(img, interpolation=interp)
    ax_img.axis('off')

    for j in range(n_panels):
        ax_img.text(
            (j + 0.012) / n_panels, 0.985, PANEL_LETTERS[j],
            transform=ax_img.transAxes, ha='left', va='top',
            fontsize=PANEL_LETTER_SIZE, fontweight='bold', color='black',
        )

    if colorbar is not None:
        cax = fig.add_axes([0.34, 0.30 * cbar_h_in / (img_h_in + cbar_h_in),
                            0.32, 0.30 * cbar_h_in / (img_h_in + cbar_h_in)])
        norm = Normalize(vmin=colorbar['vmin'], vmax=colorbar['vmax'])
        cb = plt.colorbar(
            plt.cm.ScalarMappable(norm=norm, cmap=colorbar.get('cmap', DIV_CMAP)),
            cax=cax, orientation='horizontal',
        )
        cb.set_label(colorbar['label'], fontsize=8)
        cb.ax.tick_params(labelsize=7, width=0.6, length=2.5)
        cb.outline.set_linewidth(0.6)

    paths = _save_current(fig, out_stem, vector=False)
    plt.close(fig)
    if number is not None:
        register_figure(number, out_stem, paths, title, legend, alt_text)
    return paths[0]


def add_panel_letters_to_axes(axes, offset=(-0.02, 1.06)):
    """Panel letters for pure matplotlib multi-panel figures."""
    flat = np.asarray(axes).ravel()
    for k, ax in enumerate(flat):
        ax.text(offset[0], offset[1], PANEL_LETTERS[k], transform=ax.transAxes,
                ha='left', va='bottom', fontsize=PANEL_LETTER_SIZE, fontweight='bold')


def compute_robust_clim(grid, scalar_name, pct=STRAIN_CLIM_PCT,
                        basal_exclude=STRAIN_BASAL_EXCLUDE):
    """Symmetric clim from a mass-of-wall percentile, excluding the basal rim."""
    vals = np.asarray(grid.cell_data[scalar_name])
    cc = grid.cell_centers().points
    zmin, zmax = cc[:, 2].min(), cc[:, 2].max()
    cutoff = zmax - basal_exclude * (zmax - zmin)
    keep = cc[:, 2] <= cutoff
    subset = vals[keep] if keep.any() else vals
    lo, hi = np.nanpercentile(subset, pct)
    vmax = max(abs(lo), abs(hi))
    if vmax == 0:
        vmax = 1.0
    return (-float(vmax), float(vmax))


def compute_sym_clim(values, pct=(2, 98), center_zero=True):
    """Legacy helper kept for external callers. Prefer compute_robust_clim."""
    vals = np.asarray(values)
    lo, hi = np.nanpercentile(vals, pct)
    if center_zero:
        vmax = max(abs(lo), abs(hi))
        if vmax == 0:
            vmax = 1.0
        return (-float(vmax), float(vmax))
    return (float(lo), float(hi))



## Figure-generation functions


In [ ]:
# ================================
# 5) Figure-generation functions
# ================================
# Every figure below is drawn on the END-DIASTOLIC mesh (`grid_ed`), which is
# now the reference configuration, and every strain field is the ED-referred
# quantity computed in section 3b.

_MEASURE_SYMBOL = {'inverse_gl': 'E^{*}', 'almansi': 'e'}[STRAIN_MEASURE_OUT]
_MEASURE_WORD = {
    'inverse_gl': 'end-diastolic reference, end diastole to early diastole',
    'almansi': 'end-diastolic reference, Almansi-Euler',
}[STRAIN_MEASURE_OUT]

STRAIN_LABELS_ED = {
    'circ': rf'Circumferential strain ${_MEASURE_SYMBOL}_{{cc}}$ (dimensionless)',
    'rad':  rf'Radial strain ${_MEASURE_SYMBOL}_{{rr}}$ (dimensionless)',
    'lon':  rf'Longitudinal strain ${_MEASURE_SYMBOL}_{{ll}}$ (dimensionless)',
}
STRAIN_WORD = {'circ': 'circumferential', 'rad': 'radial', 'lon': 'longitudinal'}

_REF_SENTENCE = (
    'The end-diastolic configuration is the reference configuration; strains '
    'are the Green-Lagrange strains of the inverse map from end diastole to '
    'early diastole, obtained from the inverse deformation gradient '
    r'$\mathbf{F}^{-1}$, so that end diastole is the zero-strain state.'
) if STRAIN_MEASURE_OUT == 'inverse_gl' else (
    'The end-diastolic configuration is the reference configuration; strains '
    'are Almansi-Euler strains of the early-diastole to end-diastole map, '
    'measured per unit end-diastolic length.'
)


def figure_geometry_plain(grid):
    def draw(pl, ds):
        surf = extract_surface(ds)
        add_plain_surface(pl, surf, show_wire=True)

    img = dual_view_image(grid, draw)
    return compose_panels(
        img, OUTPUT_DIR / 'fig_01_geometry_plain', n_panels=2, number=1,
        title='Left-ventricular finite-element geometry in the end-diastolic '
              '(reference) configuration.',
        legend='(A) Exterior surface with the surface wireframe overlaid. '
               '(B) Long-axis cutaway through the mid-cavity plane, showing the '
               'wall thickness. Both panels show the same hexahedral mesh in the '
               'end-diastolic configuration. The black outline is the outer '
               'silhouette of the ventricle.',
        alt_text='Two greyscale renderings of a left-ventricular mesh shaped like '
                 'a truncated ellipsoid. Panel A shows the outer surface with a '
                 'fine quadrilateral wireframe. Panel B shows the same shape cut '
                 'along its long axis, exposing the cavity and the surrounding '
                 'wall of roughly uniform thickness.',
    )


def figure_aha17_boundaries(grid):
    def draw(pl, ds):
        surf = extract_surface(ds)
        add_plain_surface(pl, surf, show_wire=False)
        add_edges(pl, extract_label_boundaries(surf, 'AHA17'),
                  AHA17_EDGE_COLOR, AHA17_EDGE_WIDTH, AHA17_EDGE_ALPHA)

    img = triple_view_image(grid, draw)
    return compose_panels(
        img, OUTPUT_DIR / 'fig_02_aha17_boundaries', n_panels=3, number=2,
        title='American Heart Association 17-segment division of the '
              'left-ventricular mesh in the end-diastolic configuration.',
        legend='(A) Exterior view. (B) Long-axis cutaway. (C) Short-axis view of a '
               'basal slab seen from the base. Dark lines are the boundaries '
               'between the 17 segments; no fill colouring is applied. Segments '
               'are numbered basal inferoseptal (1) to apical cap (17) following '
               'the standard AHA convention.',
        alt_text='Three views of a left-ventricular mesh in neutral grey with dark '
                 'lines dividing the surface into segments. Panel A is an exterior '
                 'oblique view, panel B a long-axis cut, and panel C a ring-shaped '
                 'short-axis slab viewed from the base, on which the segment '
                 'boundaries divide the ring into circumferential sectors.',
    )


def figure_aha5_grouped(grid):
    def draw(pl, ds):
        surf = extract_surface(ds)
        add_categorical_surface(pl, surf, 'AHA5', AHA5_CMAP, (1, 5))
        add_edges(pl, extract_label_boundaries(surf, 'AHA17'),
                  AHA17_EDGE_COLOR, AHA17_EDGE_WIDTH, AHA17_EDGE_ALPHA)
        add_edges(pl, extract_label_boundaries(surf, 'AHA5'),
                  AHA5_EDGE_COLOR, AHA5_EDGE_WIDTH, AHA5_EDGE_ALPHA)

    def labels(pl, ds, panel_index):
        if panel_index in (0, 2):
            add_region_labels(pl, ds, 'AHA5', AHA5_GROUP_NAMES)

    img = triple_view_image(grid, draw, label_fn=labels)
    return compose_panels(
        img, OUTPUT_DIR / 'fig_03_aha5_grouped', n_panels=3, number=3,
        title='Five-zone regional division of the left ventricle used for '
              'parameter inference, shown on the end-diastolic configuration.',
        legend='(A) Exterior view. (B) Long-axis cutaway. (C) Short-axis basal '
               'slab. The five zones are anterior, lateral, inferior, septal and '
               'apical; each zone is filled with a distinct colour and named '
               'directly on panels A and C. Thick black lines are the zone '
               'boundaries and thin dark lines are the underlying AHA 17-segment '
               'boundaries. The zones are obtained by grouping AHA segments 3 and '
               '9 (anterior); 4, 5, 10 and 11 (lateral); 6 and 12 (inferior); '
               '1, 2, 7 and 8 (septal); and 13 to 17 (apical).',
        alt_text='Three views of a left-ventricular mesh whose surface is divided '
                 'into five differently coloured zones, each annotated with its '
                 'name: anterior, lateral, inferior, septal and apical. Thick black '
                 'lines separate the zones and thinner lines show the finer '
                 '17-segment division. Panel C, a short-axis ring seen from the '
                 'base, shows the septal zone opposite the lateral zone and the '
                 'anterior zone opposite the inferior zone.',
    )


def figure_inverse_displacement(grid_ed, grid_ed0, dis):
    """ED (solid) versus early diastole recovered through the inverse map."""
    mag = np.linalg.norm(dis, axis=1)
    center = np.array(grid_ed.center)
    ed_clip = grid_ed.clip(normal=CLIP_NORMAL, origin=center, invert=False)
    ed0_clip = grid_ed0.clip(normal=CLIP_NORMAL, origin=center, invert=False)
    camera = compute_camera(grid_ed.bounds, view='oblique')

    window_size = render_window_size(2, panel_aspect=1.0)
    pl = pv.Plotter(shape=(1, 2), off_screen=True, window_size=window_size, border=False)
    pl.set_background('white')
    for j, (g_ed, g_ed0) in enumerate([(grid_ed, grid_ed0), (ed_clip, ed0_clip)]):
        pl.subplot(0, j)
        if PARALLEL_PROJ:
            pl.enable_parallel_projection()
        s_ed = extract_surface(g_ed)
        s_ed0 = extract_surface(g_ed0)
        pl.add_mesh(s_ed, color='#666666', smooth_shading=True, opacity=0.85,
                    ambient=0.30, diffuse=0.70, specular=0.05,
                    silhouette=dict(color='#222222', line_width=2.0))
        pl.add_mesh(s_ed0, style='wireframe', color='#B0B0B0', line_width=0.8,
                    opacity=0.9, lighting=False)
        pl.camera_position = camera
        pl.reset_camera()
        pl.camera.zoom(ZOOM)
    img = pl.screenshot(return_img=True)
    pl.close()

    return compose_panels(
        img, OUTPUT_DIR / 'fig_04_inverse_displacement', n_panels=2, number=4,
        title='Displacement field of the inverse map from end diastole to early '
              'diastole.',
        legend='(A) Exterior view. (B) Long-axis cutaway. The solid grey surface is '
               'the end-diastolic configuration, which is the reference '
               'configuration here; the light wireframe is the early-diastolic '
               'configuration recovered by applying the inverse displacement field '
               r'$-\mathbf{u}$ to every node. Mean and maximum nodal displacement '
               f'magnitudes are {mag.mean():.3f} and {mag.max():.3f} mesh length '
               'units respectively. The wireframe lies inside the solid surface '
               'throughout, reflecting the volume reduction on unloading.',
        alt_text='Two renderings of a left-ventricular mesh. In each, a solid grey '
                 'surface representing the end-diastolic shape encloses a lighter '
                 'wireframe representing the smaller early-diastolic shape. Panel A '
                 'is an exterior view and panel B a long-axis cut in which the '
                 'wireframe cavity is visibly smaller than the solid one.',
    )


def figure_strain(grid, direction, grouped=False):
    scalar = f'strain_ed_{direction}'
    clim = compute_robust_clim(grid, scalar)

    def draw(pl, ds):
        surf = extract_surface(ds)
        add_continuous_surface(pl, surf, scalar, clim)
        if grouped:
            add_edges(pl, extract_label_boundaries(surf, 'AHA17'),
                      AHA17_EDGE_COLOR, AHA17_EDGE_WIDTH, AHA17_EDGE_ALPHA)
            add_edges(pl, extract_label_boundaries(surf, 'AHA5'),
                      AHA5_EDGE_COLOR, AHA5_EDGE_WIDTH, AHA5_EDGE_ALPHA)

    img = dual_view_image(grid, draw)

    number_map = {
        ('circ', False): 5, ('rad', False): 6, ('lon', False): 7,
        ('circ', True): 8, ('rad', True): 9, ('lon', True): 10,
    }
    num = number_map[(direction, grouped)]
    suffix = 'grouped' if grouped else 'full'
    stem = OUTPUT_DIR / f'fig_{num:02d}_strain_ed_{direction}_{suffix}'
    word = STRAIN_WORD[direction]
    lo, hi = clim
    boundary_txt = (
        ' Thick black lines mark the five-zone boundaries and thin dark lines the '
        'AHA 17-segment boundaries.' if grouped else ''
    )
    return compose_panels(
        img, stem, n_panels=2, number=num,
        colorbar={'vmin': lo, 'vmax': hi, 'label': STRAIN_LABELS_ED[direction]},
        title=f'{word.capitalize()} strain field of the left ventricle referred to '
              f'the end-diastolic configuration'
              f'{", with the regional division overlaid" if grouped else ""}.',
        legend='(A) Exterior view. (B) Long-axis cutaway. ' + _REF_SENTENCE +
               ' Colour encodes the element-wise ' + word + ' strain component, '
               'which is dimensionless; the diverging scale is symmetric about '
               f'zero and truncated at {lo:+.3f} and {hi:+.3f}, the symmetric '
               f'{STRAIN_CLIM_PCT[0]:.0f}th to {STRAIN_CLIM_PCT[1]:.0f}th '
               'percentile range of the wall excluding a thin band of elements at '
               'the basal cut face. Values outside this range are shown at the '
               'extreme colours.' + boundary_txt,
        alt_text=f'Two renderings of a left-ventricular mesh coloured by {word} '
                 'strain using a blue-to-red diverging scale, with a horizontal '
                 'colour bar beneath. Panel A is an exterior view and panel B a '
                 'long-axis cutaway showing the transmural variation of the '
                 'strain through the wall.',
    )


def figure_lateral_strain(grid, direction):
    idx = np.flatnonzero(np.isin(grid.cell_data['AHA17'], LATERAL_SEGMENTS))
    lat = grid.extract_cells(idx)
    scalar = f'strain_ed_{direction}'
    clim = compute_robust_clim(grid, scalar)   # full-grid clim keeps the 3 comparable

    def draw(pl, ds):
        surf = extract_surface(ds)
        add_continuous_surface(pl, surf, scalar, clim)
        add_edges(pl, extract_label_boundaries(surf, 'AHA17'), AHA17_EDGE_COLOR, 1.4, 0.98)

    img = dual_view_image(lat, draw)
    num_map = {'circ': 11, 'rad': 12, 'lon': 13}
    num = num_map[direction]
    stem = OUTPUT_DIR / f'fig_{num:02d}_lateral_ed_{direction}'
    word = STRAIN_WORD[direction]
    lo, hi = clim
    return compose_panels(
        img, stem, n_panels=2, number=num,
        colorbar={'vmin': lo, 'vmax': hi, 'label': STRAIN_LABELS_ED[direction]},
        title=f'{word.capitalize()} strain within the four subsegments of the '
              'lateral zone, referred to the end-diastolic configuration.',
        legend='(A) Exterior view of the extracted lateral zone. (B) Cutaway of the '
               'same subset. ' + _REF_SENTENCE + ' The subset comprises AHA '
               f'segments {", ".join(str(s) for s in LATERAL_SEGMENTS[:-1])} and '
               f'{LATERAL_SEGMENTS[-1]}, separated by thin dark lines. Colour '
               'limits are those of the whole-ventricle field, so the three '
               'lateral-zone figures are directly comparable.',
        alt_text=f'Two renderings of a curved wall patch extracted from the lateral '
                 f'wall of the left ventricle, coloured by {word} strain on a '
                 'blue-to-red diverging scale with a colour bar beneath. Thin lines '
                 'divide the patch into four subsegments.',
    )


def extract_lateral_data(grid):
    aha17 = np.asarray(grid.cell_data['AHA17'])
    data = {}
    for seg in LATERAL_SEGMENTS:
        mask = aha17 == seg
        if not mask.any():
            raise ValueError(f'Segment {seg} has no cells.')
        data[seg] = {
            key: np.asarray(grid.cell_data[f'strain_ed_{key}'])[mask]
            for key in STRAIN_COLS
        }
    return data


def plot_lateral_kde_grid(data):
    dirs = ['circ', 'rad', 'lon']
    segs = LATERAL_SEGMENTS
    fig_w = mm_to_in(FIG_WIDTH_MM)
    fig, axes = plt.subplots(len(dirs), len(segs),
                             figsize=(fig_w, fig_w * 0.62),
                             sharex='row', facecolor='white')

    for i, direction in enumerate(dirs):
        all_vals = np.concatenate([data[s][direction] for s in segs])
        lo, hi = np.percentile(all_vals, [2.5, 97.5])
        pad = 0.05 * (hi - lo if hi > lo else 1.0)
        xx = np.linspace(lo - pad, hi + pad, 400)

        for j, seg in enumerate(segs):
            ax = axes[i, j]
            vals = data[seg][direction]
            color = LATERAL_SUB_COLORS[j]

            if len(vals) > 5 and np.std(vals) > 1e-10:
                kde = gaussian_kde(vals)
                yy = kde(xx)
                ax.fill_between(xx, yy, color=color, alpha=0.55)
                ax.plot(xx, yy, color=color, lw=0.9)

            ax.axvline(vals.mean(), color='#222222', ls='--', lw=0.7)
            ax.axvline(0, color='#999999', lw=0.6)
            ax.set_yticks([])
            for side in ('top', 'right', 'left'):
                ax.spines[side].set_visible(False)
            if i == 0:
                ax.set_title(f'AHA segment {seg}', fontsize=8, color=color)
            if j == 0:
                ax.set_ylabel(f'{STRAIN_WORD[direction].capitalize()}\nkernel density',
                              fontsize=7)
            if i == len(dirs) - 1:
                ax.set_xlabel(f'${_MEASURE_SYMBOL}$ (dimensionless)', fontsize=7)

    add_panel_letters_to_axes(axes, offset=(-0.05, 0.99))
    fig.tight_layout(pad=0.4)
    stem = OUTPUT_DIR / 'fig_14_lateral_ed_distribution_kde'
    paths = _save_current(fig, stem, vector=True)
    plt.close(fig)
    register_figure(
        14, stem, paths,
        title='Kernel density estimates of the end-diastole-referred strain '
              'components within the four lateral subsegments.',
        legend='Panels A to D, E to H and I to L show the circumferential, radial '
               'and longitudinal components respectively; within each row the '
               'columns are AHA segments '
               + ', '.join(str(s) for s in LATERAL_SEGMENTS) + '. '
               + _REF_SENTENCE +
               ' Each density is estimated by a Gaussian kernel over the element '
               'values of that subsegment; the dashed vertical line is the '
               'subsegment mean and the thin solid vertical line marks zero '
               'strain. The horizontal axis is dimensionless strain and is shared '
               'within each row; the vertical axis is probability density on an '
               'arbitrary scale and is therefore unlabelled.',
        alt_text='A grid of twelve small density plots arranged in three rows and '
                 'four columns. Rows correspond to circumferential, radial and '
                 'longitudinal strain; columns to four lateral subsegments. Each '
                 'panel shows a filled unimodal density curve with a dashed line '
                 'at its mean and a reference line at zero. The circumferential '
                 'and longitudinal densities lie on one side of zero and the '
                 'radial densities on the other.',
    )
    return paths[0]


def plot_lateral_boxplot(data):
    dirs = ['circ', 'rad', 'lon']
    segs = LATERAL_SEGMENTS
    fig_w = mm_to_in(FIG_WIDTH_MM)
    fig, axes = plt.subplots(1, 3, figsize=(fig_w, fig_w * 0.38), facecolor='white')

    for i, direction in enumerate(dirs):
        ax = axes[i]
        vals = [data[s][direction] for s in segs]
        bp = ax.boxplot(vals, patch_artist=True, widths=0.6, showfliers=False)
        for patch, color in zip(bp['boxes'], LATERAL_SUB_COLORS):
            patch.set(facecolor=color, alpha=0.55, edgecolor='#444444', linewidth=0.6)
        for med in bp['medians']:
            med.set(color='#111111', linewidth=1.0)
        for element in ('whiskers', 'caps'):
            for art in bp[element]:
                art.set(linewidth=0.6, color='#444444')

        ax.axhline(0, color='#999999', lw=0.6)
        ax.set_xticks(range(1, len(segs) + 1))
        ax.set_xticklabels([str(s) for s in segs])
        ax.set_xlabel('AHA 17 segment', fontsize=7)
        ax.set_ylabel(f'{STRAIN_WORD[direction].capitalize()} strain '
                      f'${_MEASURE_SYMBOL}$ (dimensionless)', fontsize=7)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)

    add_panel_letters_to_axes(axes, offset=(-0.14, 1.02))
    fig.tight_layout(pad=0.4)
    stem = OUTPUT_DIR / 'fig_15_lateral_ed_distribution_boxplot'
    paths = _save_current(fig, stem, vector=True)
    plt.close(fig)
    register_figure(
        15, stem, paths,
        title='Distribution of the end-diastole-referred strain components across '
              'the four lateral subsegments.',
        legend='(A) Circumferential, (B) radial and (C) longitudinal strain. '
               + _REF_SENTENCE +
               ' Each box spans the interquartile range of the element values of '
               'one AHA segment, the central line is the median and the whiskers '
               'extend to 1.5 times the interquartile range; outliers are not '
               'drawn. Boxes are shaded by subsegment and identified by the '
               'segment number on the horizontal axis. The horizontal grey line '
               'marks zero strain. Strain is dimensionless.',
        alt_text='Three box-plot panels side by side for circumferential, radial '
                 'and longitudinal strain. Each panel contains four boxes, one per '
                 'lateral subsegment, drawn against a dimensionless strain axis '
                 'with a grey line at zero. The circumferential and longitudinal '
                 'boxes sit on one side of zero and the radial boxes on the other, '
                 'with modest differences in median between subsegments.',
    )
    return paths[0]


## Load the data and switch the reference configuration

- `points_ed0` / `grid_ed0`: early diastole, the solver's unloaded reference
- `points_ed` / `grid_ed`: **end diastole — the reference configuration used from here on**
- `F`: element-centroid deformation gradient of the early-diastole $\to$ end-diastole map
- `strain_ed`: ED-referred normal strain components obtained from $\mathbf{F}^{-1}$

The audit printout compares the two independent routes to the ED-referred strain and tests the Voigt layout of the stored tensor. Read it before using any figure.


In [ ]:
# ==========================================================================
# 6) Load all data, switch the reference configuration to end diastole
# ==========================================================================

points, conn = parse_abaqus_inp(INP_PATH)
n_points = points.shape[0]
n_cells = conn.shape[0]
print(f'Mesh: {n_points} points, {n_cells} cells')

node_regions, aha17, aha5, conflict_mask = load_aha_labels(AHA_MAT, n_cells)
print(f'Conflict cells reassigned to segment 17: {int(conflict_mask.sum())}')

# Data-cleaning step: Ant/Lat labels at the Septal-Inferior boundary are wrong.
_cell_centers_tmp = points[conn].mean(axis=1)
aha17, aha5, _n_fixed = fix_septal_inferior_boundary(
    aha17, aha5, _cell_centers_tmp,
    inf_theta_min=30.0, inf_theta_max=90.0,
)
del _cell_centers_tmp

S = load_strain_struct(STRAIN_MAT, n_cells_expected=n_cells, n_points_expected=n_points)
print(f'strain_tensor_cra shape: {S["strain_tensor"].shape}')
print(f'dis shape: {S["dis"].shape}')

max_abs_point_diff = float(np.abs(points - S['node_ori']).max())
print(f'Max |points - node_ori| = {max_abs_point_diff:.6e}')

print('\nStored (early-diastole-referred) normal strains:')
for key in ('circ', 'rad', 'lon'):
    col = S['strain_tensor'][:, STRAIN_COLS[key]]
    print(f"  {key:>5s} <- col {STRAIN_COLS[key]}:  mean={col.mean():+.4f}  "
          f"median={np.median(col):+.4f}  std={col.std():.4f}  "
          f"range=[{col.min():+.4f}, {col.max():+.4f}]")

# --- Coordinates of the two configurations -------------------------------
points_ed0 = points                      # early diastole (unloaded)
points_ed = points + S['dis']            # end diastole  <- new reference

# --- Deformation gradient and the audit ----------------------------------
F = deformation_gradient_hex(points_ed0, conn, S['dis'])
cell_centers_ed = points_ed[conn].mean(axis=1)

print('\n' + '=' * 72)
print('Kinematic audit (read this before using any figure)')
print('=' * 72)
_co, _ge = audit_kinematics(S['strain_tensor'], F, cell_centers_ed)

# --- ED-referred strain components ---------------------------------------
strain_ed = ed_strain_components(S['strain_tensor'], F, cell_centers_ed)
print(f'\nED-referred strains in use (mode={STRAIN_AXES_MODE}, '
      f'measure={STRAIN_MEASURE_OUT}):')
for key in ('circ', 'rad', 'lon'):
    v = strain_ed[key]
    print(f'  {key:>5s}: mean={v.mean():+.4f}  median={np.median(v):+.4f}  '
          f'std={v.std():.4f}  range=[{v.min():+.4f}, {v.max():+.4f}]')
if STRAIN_MEASURE_OUT == 'inverse_gl':
    print('Expected signs for the end-diastole -> early-diastole map: '
          'circumferential -, radial +, longitudinal -.')

# --- Grids ----------------------------------------------------------------
# grid_ed  : the new reference configuration, carries the ED-referred strains
# grid_ed0 : early diastole, recovered by the inverse map (nodewise x -> X)
grid_ed = build_grid(points_ed, conn, node_regions, aha17, aha5,
                     S['strain_tensor'], S['dis'])
grid_ed0 = build_grid(points_ed0, conn, node_regions, aha17, aha5,
                      S['strain_tensor'], S['dis'])

for key in ('circ', 'rad', 'lon'):
    grid_ed.cell_data[f'strain_ed_{key}'] = strain_ed[key].astype(np.float64)
    grid_ed0.cell_data[f'strain_ed_{key}'] = strain_ed[key].astype(np.float64)

# Inverse displacement field carried on the ED mesh.
grid_ed.point_data['inv_disp_x'] = -S['dis'][:, 0]
grid_ed.point_data['inv_disp_y'] = -S['dis'][:, 1]
grid_ed.point_data['inv_disp_z'] = -S['dis'][:, 2]
grid_ed.point_data['inv_disp_mag'] = np.linalg.norm(S['dis'], axis=1)
grid_ed.cell_data['detF'] = np.linalg.det(F)

if SAVE_VTU:
    ed_vtu = OUTPUT_DIR / 'lv_end_diastolic_reference_with_ed_strain.vtu'
    ed0_vtu = OUTPUT_DIR / 'lv_early_diastolic_inverse_mapped.vtu'
    grid_ed.save(ed_vtu)
    grid_ed0.save(ed0_vtu)
    print(f'\nSaved VTU: {ed_vtu}')
    print(f'Saved VTU: {ed0_vtu}')

print('\nAHA5 cell counts:')
for gid in sorted(AHA5_GROUP_NAMES):
    print(f'  {gid} - {AHA5_GROUP_NAMES[gid]:9s}: {int((aha5 == gid).sum())}')
print('\nLateral subsegments used in this notebook:', LATERAL_SEGMENTS)


## Generate the geometry, segmentation and inverse-displacement figures

In [ ]:
# ================================
# 7) Geometry, segmentation, inverse displacement
# ================================

figure_geometry_plain(grid_ed)
figure_aha17_boundaries(grid_ed)
figure_aha5_grouped(grid_ed)
figure_inverse_displacement(grid_ed, grid_ed0, S['dis'])


## Generate the full-field ED-referred strain figures

In [ ]:
# ================================
# 8) Full-field ED-referred strain figures
# ================================

for direction in ['circ', 'rad', 'lon']:
    figure_strain(grid_ed, direction, grouped=False)


## Generate the ED-referred strain figures with the five-zone division

In [ ]:
# ================================
# 9) ED-referred strain with the five-zone division overlaid
# ================================

for direction in ['circ', 'rad', 'lon']:
    figure_strain(grid_ed, direction, grouped=True)


## Generate the lateral-zone subsegment strain figures

In [ ]:
# ================================
# 10) Lateral-zone subsegment strain figures
# ================================

for direction in ['circ', 'rad', 'lon']:
    figure_lateral_strain(grid_ed, direction)


## Generate the lateral-zone strain distribution figures

In [ ]:
# ================================
# 11) Lateral-zone strain distributions
# ================================

lateral_data = extract_lateral_data(grid_ed)
plot_lateral_kde_grid(lateral_data)
plot_lateral_boxplot(lateral_data)


## Composite figure: whole LV, lateral zone and the KDE grid

One-page figure combining the whole-ventricle strain rendering (A), the extracted lateral zone (B) with a shared colour bar, and the 3x4 kernel-density grid (C: rows = strain component, columns = lateral subsegments). Generated for all three components (Figs 16-18). Set the loop at the foot of the cell to a single direction if you only need one.


In [ ]:
# ==========================================================================
# 13) Composite summary figure: whole LV + lateral zone + KDE grid
# ==========================================================================
# Layout (one file, one page):
#   A  whole-LV strain rendering            |  C  3 x 4 grid of kernel density
#   B  lateral-zone strain rendering        |     estimates: rows = strain
#      shared colour bar beneath A and B    |     component, columns = AHA
#                                           |     segments 4, 5, 10, 11
#
# Journal rules kept: no figure title and no key inside the image; the two
# renderings and the density grid each carry a letter; the colour bar and all
# axes are labelled with the quantity and its (dimensionless) units.


def _trim_white(img, tol=248):
    """Crop the white border produced by the off-screen renderer."""
    a = np.asarray(img)
    mask = (a[:, :, :3] < tol).any(axis=2)
    if not mask.any():
        return a
    rows = np.flatnonzero(mask.any(axis=1))
    cols = np.flatnonzero(mask.any(axis=0))
    pad = 4
    r0 = max(rows[0] - pad, 0)
    r1 = min(rows[-1] + 1 + pad, a.shape[0])
    c0 = max(cols[0] - pad, 0)
    c1 = min(cols[-1] + 1 + pad, a.shape[1])
    return a[r0:r1, c0:c1]


def single_view_image(dataset, draw_fn, width_frac=0.28, aspect=1.25,
                      view='oblique', trim=True):
    """One rendered view, sized so it still meets DPI_RASTER in the composite.

    width_frac : fraction of the figure width the panel will occupy
    aspect     : panel height / panel width
    """
    w = int(round(mm_to_in(FIG_WIDTH_MM) * DPI_RASTER * width_frac))
    w = min(w, MAX_RENDER_PX)
    h = int(round(w * aspect))
    camera = compute_camera(dataset.bounds, view=view)
    pl = pv.Plotter(off_screen=True, window_size=(w, h), border=False)
    pl.set_background('white')
    _render_panel(pl, dataset, draw_fn, camera)
    img = np.asarray(pl.screenshot(return_img=True))
    pl.close()
    return _trim_white(img) if trim else img


def figure_lv_lateral_kde(grid, direction='circ', number=16,
                          fig_height_ratio=0.52,
                          panel_letters=True, vector=False):
    """Composite of the whole-LV field, the lateral zone and the KDE grid."""
    scalar = f'strain_ed_{direction}'
    clim = compute_robust_clim(grid, scalar)
    word = STRAIN_WORD[direction]

    # --- renderings -------------------------------------------------------
    def draw(pl, ds):
        surf = extract_surface(ds)
        add_continuous_surface(pl, surf, scalar, clim)
        add_edges(pl, extract_label_boundaries(surf, 'AHA17'),
                  AHA17_EDGE_COLOR, 1.6, 0.98)

    lat_idx = np.flatnonzero(np.isin(grid.cell_data['AHA17'], LATERAL_SEGMENTS))
    lat = grid.extract_cells(lat_idx)

    img_lv = single_view_image(grid, draw, width_frac=0.26, aspect=1.30)
    img_lat = single_view_image(lat, draw, width_frac=0.26, aspect=0.95)

    # --- data for the density grid ---------------------------------------
    data = extract_lateral_data(grid)
    dirs = ['circ', 'rad', 'lon']
    segs = list(LATERAL_SEGMENTS)

    # --- layout -----------------------------------------------------------
    fig_w = mm_to_in(FIG_WIDTH_MM)
    fig = plt.figure(figsize=(fig_w, fig_w * fig_height_ratio), facecolor='white')
    gs = fig.add_gridspec(1, 2, width_ratios=[0.30, 0.70], wspace=0.06,
                          left=0.01, right=0.99, top=0.94, bottom=0.10)
    gs_l = gs[0].subgridspec(3, 1, height_ratios=[1.00, 0.80, 0.16], hspace=0.05)
    gs_r = gs[1].subgridspec(len(dirs), len(segs), hspace=0.45, wspace=0.18)

    ax_lv = fig.add_subplot(gs_l[0])
    ax_lat = fig.add_subplot(gs_l[1])
    ax_cb = fig.add_subplot(gs_l[2])
    for ax, im in ((ax_lv, img_lv), (ax_lat, img_lat)):
        ax.imshow(im, interpolation='lanczos')
        ax.axis('off')

    ax_cb.axis('off')
    cax = ax_cb.inset_axes([0.08, 0.42, 0.84, 0.30])
    norm = Normalize(vmin=clim[0], vmax=clim[1])
    cb = plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=DIV_CMAP),
                      cax=cax, orientation='horizontal')
    cb.set_label(STRAIN_LABELS_ED[direction], fontsize=7.5)
    cb.ax.tick_params(labelsize=6.5, width=0.6, length=2.5)
    cb.outline.set_linewidth(0.6)

    axes = np.empty((len(dirs), len(segs)), dtype=object)
    for i, d in enumerate(dirs):
        for j, seg in enumerate(segs):
            axes[i, j] = fig.add_subplot(gs_r[i, j])

    for i, d in enumerate(dirs):
        all_vals = np.concatenate([data[s][d] for s in segs])
        lo, hi = np.percentile(all_vals, [1.0, 99.0])
        span = hi - lo if hi > lo else 1.0
        xlo, xhi = lo - 0.08 * span, hi + 0.08 * span
        xlo, xhi = min(xlo, 0.0), max(xhi, 0.0)     # keep zero inside the row
        xx = np.linspace(xlo, xhi, 400)

        for j, seg in enumerate(segs):
            ax = axes[i, j]
            vals = data[seg][d]
            color = LATERAL_SUB_COLORS[j]
            if len(vals) > 5 and np.std(vals) > 1e-10:
                yy = gaussian_kde(vals)(xx)
                ax.fill_between(xx, yy, color=color, alpha=0.60, lw=0)
                ax.plot(xx, yy, color=color, lw=0.8)
            ax.axvline(vals.mean(), color='#222222', ls='--', lw=0.7)
            ax.axvline(0.0, color='#999999', lw=0.6)
            ax.set_xlim(xlo, xhi)
            ax.set_ylim(bottom=0.0)
            ax.set_yticks([])
            for side in ('top', 'right', 'left'):
                ax.spines[side].set_visible(False)
            ax.tick_params(axis='x', labelsize=6, width=0.6, length=2.0, pad=1.5)
            if i == 0:
                ax.set_title(f'AHA segment {seg}', fontsize=7.5, color=color, pad=3)
            if j == 0:
                ax.set_ylabel(f'{STRAIN_WORD[dirs[i]].capitalize()}\nkernel density',
                              fontsize=7, labelpad=2)

    # shared axis label for the density grid
    fig.text(0.30 + 0.70 / 2, 0.022,
             f'Strain ${_MEASURE_SYMBOL}$ (dimensionless)',
             ha='center', va='bottom', fontsize=8)

    # panel letters (set panel_letters=False to omit them from the image)
    if panel_letters:
        for ax, letter in ((ax_lv, 'A'), (ax_lat, 'B')):
            ax.text(0.0, 1.0, letter, transform=ax.transAxes, ha='left', va='top',
                    fontsize=PANEL_LETTER_SIZE, fontweight='bold')
        axes[0, 0].text(-0.30, 1.30, 'C', transform=axes[0, 0].transAxes,
                        ha='left', va='top', fontsize=PANEL_LETTER_SIZE,
                        fontweight='bold')

    stem = OUTPUT_DIR / f'fig_{number:02d}_lv_lateral_kde_{direction}'
    if vector:                       # PDF only: vector KDE panels, raster renders embedded
        pdf = stem.with_suffix('.pdf')
        fig.savefig(pdf, facecolor='white')
        print(f'Saved: {pdf}')
        paths = [pdf]
    else:
        paths = _save_current(fig, stem, vector=False)
    plt.close(fig)

    register_figure(
        number, stem, paths,
        title=f'{word.capitalize()} strain of the left ventricle referred to the '
              'end-diastolic configuration, with its distribution across the '
              'lateral subsegments.',
        legend='(A) Whole ventricle, oblique exterior view. (B) The lateral zone '
               f'alone, comprising AHA segments {", ".join(str(s) for s in segs[:-1])} '
               f'and {segs[-1]}, in the same orientation. In (A) and (B) colour '
               f'encodes the element-wise {word} strain on the shared scale beneath, '
               f'truncated at {clim[0]:+.3f} and {clim[1]:+.3f}; thin dark lines are '
               'the AHA segment boundaries. (C) Gaussian kernel density estimates of '
               'the element values within each lateral subsegment: rows are the '
               'circumferential, radial and longitudinal components and columns are '
               'the four subsegments, identified above each column. The dashed line '
               'in each panel is the subsegment mean and the thin solid line marks '
               'zero strain; the horizontal axis is shared within a row and the '
               'vertical axis is probability density on an arbitrary scale. '
               + _REF_SENTENCE,
        alt_text='A composite figure. Panel A is an oblique rendering of a '
                 f'left-ventricular mesh coloured by {word} strain on a '
                 'blue-to-red diverging scale, with segment boundaries drawn as '
                 'thin lines and a labelled colour bar beneath. Panel B shows the '
                 'lateral wall alone in the same colours and orientation. Panel C '
                 'is a grid of twelve density curves, three rows for the '
                 'circumferential, radial and longitudinal components by four '
                 'columns for the lateral subsegments; each curve is filled, '
                 'unimodal and marked with its mean and a reference line at zero.',
    )
    return paths[0]


# Generate the composite for all three components.
for _direction, _num in (('circ', 16), ('rad', 17), ('lon', 18)):
    figure_lv_lateral_kde(grid_ed, _direction, number=_num)


## Captions, alt text and submission check

In [ ]:
# ================================
# 12a) List output files
# ================================

for p in sorted(OUTPUT_DIR.glob('*')):
    print(p)


In [ ]:
# ==========================================================================
# 12) Captions, alt text, and a submission check
# ==========================================================================
# Titles and legends are deliberately absent from the image files. They are
# written here in the order required by the journal:
#     Figure n. <title> <legend>
#     Alt text: <description>
# Paste this block into the main manuscript file.

caption_path = OUTPUT_DIR / 'figure_captions_and_alt_text.md'
lines = [
    '# Figure captions and alt text',
    '',
    'Insert each block at the appropriate point in the main manuscript file. '
    'Alt text follows the figure legend directly, as required.',
    '',
]
for item in sorted(FIGURE_REGISTRY, key=lambda d: d['number']):
    lines.append(f"**Figure {item['number']}.** {item['title']} {item['legend']}")
    lines.append('')
    lines.append(f"Alt text: {item['alt']}")
    lines.append('')
    lines.append(f"_File(s): {', '.join(item['files'])}_")
    lines.append('')
    lines.append('---')
    lines.append('')

caption_path.write_text('\n'.join(lines), encoding='utf-8')
print(f'Saved: {caption_path}\n')

# --- Submission check ----------------------------------------------------
from PIL import Image

print(f'{"file":58s} {"px":>13s} {"mm":>13s} {"dpi":>9s}')
print('-' * 96)
for item in sorted(FIGURE_REGISTRY, key=lambda d: d['number']):
    for name in item['files']:
        p = OUTPUT_DIR / name
        if p.suffix.lower() in ('.tif', '.tiff'):
            with Image.open(p) as im:
                w, h = im.size
                dpi = im.info.get('dpi', (None, None))
            dx = float(dpi[0]) if dpi[0] else 0.0
            mm = (w / dx * 25.4, h / dx * 25.4) if dx else (0, 0)
            flag = '' if (dx and dx >= 350) else '   <-- CHECK'
            print(f'{name:58s} {w:6d}x{h:<6d} {mm[0]:6.1f}x{mm[1]:<6.1f} '
                  f'{dx:9.0f}{flag}')
        else:
            print(f'{name:58s} {"vector":>13s} {FIG_WIDTH_MM:11.1f}mm {"n/a":>9s}')
print('-' * 96)
print('Rules applied: one file per figure, all panels on a single page, panel '
      'letters A, B, C ... in the upper-left corner, no titles or keys inside '
      'the image, TIFF at 600 dpi for the renderings and vector PDF for the '
      'line-art figures.')
